# 第 3–4 天：光学对照基线

**原封不动**地用光学默认设置训练，得到的分数是之后所有改进的对照组。需要 GPU 和 `pip install ultralytics`。

建议直接用命令行跑（结果会自动追加到 `reports/experiments.csv`）：

```bash
python scripts/train_baseline.py --data data/yolo_raw/data.yaml --name baseline_optical --model yolo11s.pt
```

下面的单元格做同样的事，方便边跑边看。

In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK SC", "SimHei", "PingFang SC", "Microsoft YaHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

# 下载了真实数据就改成 data/raw/UATD_Training；没下载就先用合成数据（python scripts/make_synthetic.py）
DATA_ROOT = REPO / "data" / "raw" / "UATD_Training"
if not DATA_ROOT.exists():
    DATA_ROOT = REPO / "data" / "synthetic" / "UATD_Training"
print("使用数据：", DATA_ROOT)

In [ ]:
from ultralytics import YOLO
DATA = REPO / "data" / "yolo_raw" / "data.yaml"
model = YOLO("yolo11s.pt")            # COCO 预训练
model.train(data=str(DATA), epochs=100, imgsz=640, batch=16, seed=0, deterministic=True,
            project=str(REPO / "runs"), name="baseline_optical", exist_ok=True)

## 在验证集（和测试集）上评估

In [ ]:
import yaml
best = YOLO(str(REPO / "runs" / "baseline_optical" / "weights" / "best.pt"))
splits = ["val"] + (["test"] if "test" in yaml.safe_load(DATA.read_text()) else [])
res = {s: best.val(data=str(DATA), split=s, project=str(REPO / "runs"), name=f"baseline_{s}", exist_ok=True) for s in splits}
for s, m in res.items():
    print(f"{s}: mAP50={m.box.map50:.4f}  mAP50-95={m.box.map:.4f}")

## 各类别 AP：哪个类最差？

In [ ]:
m = res[splits[-1]]
per_cls = pd.Series(m.box.maps, index=[m.names[i] for i in range(len(m.box.maps))], name="AP50-95").sort_values()
per_cls.plot.barh(figsize=(6, 3.5), title=f"各类别 AP50-95（{splits[-1]}）"); plt.tight_layout(); plt.show()
per_cls

> 问题：
> 1. 验证集和测试集的分数差多少？差得多说明什么？
> 2. 最差的两个类别，在第 1 天的统计里是不是样本最少或者框最小？
> 3. 把这组数字填进 README 的实验表第一行，标注「光学默认设置」。